# 01 — Ingestão Bronze BigQuery

Este notebook realiza **somente a ingestão da camada Bronze** do projeto FIAP Tech Challenge Fase 2.

Fluxo executado:

1. Lê as configurações do arquivo `.env`.
2. Conecta no Google BigQuery.
3. Define as consultas das tabelas da Base dos Dados.
4. Executa `dry run` para estimar volume processado.
5. Baixa as tabelas selecionadas uma única vez.
6. Salva os arquivos em `data/bronze/` no formato Parquet.

> Importante: por segurança, a tabela `alunos` fica comentada inicialmente, pois pode ter maior volume.

## 1. Imports e configuração

In [ ]:
# gcloud auth application-default revoke
# gcloud auth application-default login
# gcloud auth application-default set-quota-project fiap-tech-challenge-fase2
# gcloud auth application-default print-access-token

In [ ]:
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
MAX_BYTES_BILLED = int(os.getenv("BQ_MAX_BYTES_BILLED", "100000000"))

if not PROJECT_ID:
    raise ValueError("GCP_PROJECT_ID não encontrado. Verifique o arquivo .env na raiz do projeto.")

client = bigquery.Client(project=PROJECT_ID)

print("Projeto GCP:", PROJECT_ID)
print("Limite máximo por consulta:", f"{MAX_BYTES_BILLED / 1024 / 1024:.2f} MB")

## 2. Parâmetros da camada Bronze

In [ ]:
execution_date = datetime.now().strftime("%Y-%m-%d")

BASE_PATH = Path("../data")
BRONZE_BASE_PATH = BASE_PATH / "bronze"

BRONZE_BASE_PATH.mkdir(parents=True, exist_ok=True)

print("Data de execução:", execution_date)
print("Base Bronze:", BRONZE_BASE_PATH)

## 3. Queries da camada Bronze

As queries abaixo buscam as entidades necessárias para o projeto:

- `uf`
- `meta_alfabetizacao_brasil`
- `meta_alfabetizacao_uf`
- `meta_alfabetizacao_municipio`
- `municipio`
- `alunos`

As tabelas temporárias criadas em `WITH` são usadas sem crase nos `JOINs`.

In [ ]:
def carregar_query(caminho_query: str) -> str:
    with open(caminho_query, "r", encoding="utf-8") as arquivo:
        return arquivo.read()


queries = {
    "uf": carregar_query("../queries/bronze/uf.sql"),
    "meta_alfabetizacao_brasil": carregar_query("../queries/bronze/meta_alfabetizacao_brasil.sql"),
    "meta_alfabetizacao_uf": carregar_query("../queries/bronze/meta_alfabetizacao_uf.sql"),
    "meta_alfabetizacao_municipio": carregar_query("../queries/bronze/meta_alfabetizacao_municipio.sql"),
    "municipio": carregar_query("../queries/bronze/municipio.sql"),
    "alunos": carregar_query("../queries/bronze/alunos.sql"),
}

print("Quantidade de consultas configuradas:", len(queries))
print("Tabelas:", list(queries.keys()))

## 4. Dry run de todas as consultas

Esta etapa estima o volume processado **sem baixar dados**.

Verifique principalmente a tabela `alunos`, pois tende a ser a maior.

In [ ]:
estimativas = []

for nome_tabela, query in queries.items():
    job_config = bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False
    )

    dry_run_job = client.query(query, job_config=job_config)

    bytes_processados = dry_run_job.total_bytes_processed
    mb_processados = bytes_processados / 1024 / 1024

    estimativas.append({
        "tabela": nome_tabela,
        "bytes_estimados": bytes_processados,
        "mb_estimados": round(mb_processados, 2)
    })

df_estimativas = pd.DataFrame(estimativas)

df_estimativas

## 5. Definir tabelas que serão baixadas

Por segurança, a tabela `alunos` fica comentada inicialmente.

Após avaliar o `dry run`, remova o `#` da linha `"alunos"` caso decida baixar essa tabela.

In [ ]:
tabelas_para_baixar = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    # "alunos",
]

print("Tabelas selecionadas para ingestão:")
for tabela in tabelas_para_baixar:
    print("-", tabela)

## 6. Função para salvar arquivos na Bronze

A função abaixo evita nova execução no BigQuery caso o arquivo local já exista.

In [ ]:
def salvar_bronze(nome_tabela: str, query: str) -> dict:
    tabela_path = BRONZE_BASE_PATH / nome_tabela / f"execution_date={execution_date}"
    tabela_path.mkdir(parents=True, exist_ok=True)

    output_file = tabela_path / f"{nome_tabela}.parquet"

    if output_file.exists():
        print(f"[SKIP] {nome_tabela}: arquivo já existe. Consulta não executada.")

        df_local = pd.read_parquet(output_file)

        return {
            "tabela": nome_tabela,
            "status": "ja_existia",
            "arquivo": str(output_file),
            "linhas": len(df_local),
            "colunas": len(df_local.columns)
        }

    print(f"[RUN] {nome_tabela}: executando consulta...")

    job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=MAX_BYTES_BILLED,
        use_query_cache=True
    )

    query_job = client.query(query, job_config=job_config)
    df = query_job.to_dataframe()

    df.to_parquet(output_file, index=False)

    print(f"[OK] {nome_tabela}: salvo em {output_file}")
    print(f"     linhas: {len(df)} | colunas: {len(df.columns)}")

    return {
        "tabela": nome_tabela,
        "status": "baixado",
        "arquivo": str(output_file),
        "linhas": len(df),
        "colunas": len(df.columns)
    }

## 7. Executar ingestão Bronze

In [ ]:
resultados = []

for tabela in tabelas_para_baixar:
    resultado = salvar_bronze(
        nome_tabela=tabela,
        query=queries[tabela]
    )

    resultados.append(resultado)

df_resultados = pd.DataFrame(resultados)

df_resultados

## 8. Conferência dos arquivos Bronze

In [ ]:
arquivos_bronze = list(BRONZE_BASE_PATH.rglob("*.parquet"))

print("Arquivos Parquet encontrados na Bronze:", len(arquivos_bronze))

for arquivo in arquivos_bronze:
    print(arquivo)

## 9. Resumo final da ingestão Bronze

In [ ]:
print("Resumo da Ingestão Bronze")
print("=" * 50)

print(f"Data de execução: {execution_date}")
print(f"Quantidade de tabelas configuradas: {len(queries)}")
print(f"Quantidade de tabelas baixadas/verificadas: {len(tabelas_para_baixar)}")

display(df_resultados)

print("Status: ingestão Bronze concluída.")